# cinematlas quickstart

Index a video, ask it a question, and jump to the second where the answer is. You need a MongoDB Atlas cluster (the free tier works) and a Voyage AI key.

In [ ]:
%pip install -q "cinematlas[video,whisper]"

In [ ]:
import os, getpass
for key in ("MONGODB_URI", "VOYAGE_API_KEY"):
    if not os.getenv(key):
        os.environ[key] = getpass.getpass(key + ": ")

In [ ]:
from cinematlas import Cinematlas

eng = Cinematlas(db_name="cinematlas_quickstart", collection_name="scenes")
eng.ensure_indexes()  # idempotent; the first run waits for Atlas to build the search indexes

Ingest a public-domain NASA interview. Scene detection, keyframes, speech-to-text, and Voyage embeddings all happen here.

In [ ]:
VIDEO = "https://images-assets.nasa.gov/video/NHQ20210805ARMD01/NHQ20210805ARMD01~small.mp4"
result = eng.ingest(VIDEO, progress=lambda stage, info: print(f"  ✓ {stage:<12} {info}"))
print(result)
result.stages  # seconds spent in each stage

Atlas syncs new documents into its search indexes within a few seconds. Wait for that, then search.

In [ ]:
import time
while len(eng.search("the").only("transcript").video(result.video_id).limit(500)) < result.spoken_scenes:
    time.sleep(2)

hits = eng.search("what did he enjoy about flying?").video(result.video_id).limit(3)
hits  # runs on first read; renders as a table, each row links to the exact second

Check what the session sent to Voyage. cinematlas ships no price table, so pass current prices from https://docs.voyageai.com/docs/pricing to estimate cost.

In [ ]:
print(eng.usage)
# eng.usage.cost({"voyage-multimodal-3.5": {"per_m_tokens": ..., "per_b_pixels": ...}})